<a href="https://colab.research.google.com/github/ryan-merser/ST554-BigData/blob/main/HW7/ST554_HW7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ST554 Analysis of Big Data Homework 7

Ryan Mersereau

In this notebook, we will get some practice fitting MLR and logistic regression models using the wine dataset found from the UCI machine learning repository  [ here.](https://archive.ics.uci.edu/dataset/186/wine+quality)

The data description describes the following variables:
1. - fixed acidity
2. - volatile acidity
3. - citric acid
4. - residual sugar
5. - chlorides
6. - free sulfur dioxide
7. - total sulfur dioxide
8. - density
9. - pH
10. - sulphates
11. - alcohol
12. - quality (score between 0 and 10)

In this notebook we'll be using alcohol as our target variable for multiple linear regression, and for fitting logistic regression type models we’ll use the type of wine as the response variable.

## Reading in and combining data

We'll start by uploading each dataset via a local path, and then combining them and create a new variable that represents the type of wine (red or white).

In [1]:
import pandas as pd
import numpy as np
red = pd.read_csv("winequality-red.csv", sep=';')
white = pd.read_csv("winequality-white.csv", sep=';')

In [2]:
red.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


In [3]:
white.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.0010,3.00,0.45,8.8,6
1,6.3,0.30,0.34,1.6,0.049,14.0,132.0,0.9940,3.30,0.49,9.5,6
2,8.1,0.28,0.40,6.9,0.050,30.0,97.0,0.9951,3.26,0.44,10.1,6
3,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6
4,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6


In [4]:
# Add type variable
red['type'] = 'red'
white['type'] = 'white'

# Combine
total_wine = pd.concat([red, white], ignore_index= True)

In [5]:
total_wine.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6497 entries, 0 to 6496
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         6497 non-null   float64
 1   volatile acidity      6497 non-null   float64
 2   citric acid           6497 non-null   float64
 3   residual sugar        6497 non-null   float64
 4   chlorides             6497 non-null   float64
 5   free sulfur dioxide   6497 non-null   float64
 6   total sulfur dioxide  6497 non-null   float64
 7   density               6497 non-null   float64
 8   pH                    6497 non-null   float64
 9   sulphates             6497 non-null   float64
 10  alcohol               6497 non-null   float64
 11  quality               6497 non-null   int64  
 12  type                  6497 non-null   object 
dtypes: float64(11), int64(1), object(1)
memory usage: 660.0+ KB


## Split data

Now, lets split up the data set into a training and test set. For this, I want you to use stratified sampling to
make sure that you have a similar proportion of white and red wines in the training and test sets. This
can be done with the `train_test_split()` function.

We also need to encode our categorical variable for type of wine as a binary variable (0 or 1) to avoid issues.

In [6]:
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression, Lasso, LassoCV
from sklearn import preprocessing

# Encode type as binary
enc = preprocessing.OrdinalEncoder()
total_wine[['type']] = enc.fit_transform(total_wine[['type']])
print(enc.categories_)

[array(['red', 'white'], dtype=object)]


In [7]:
X = total_wine.drop(columns=['alcohol'])
y = total_wine['alcohol']

# Create test/train split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2, # Typically want an 80/20 split, train on 80%, test on 20%
    random_state = 7,
    stratify = X['type']
)


In [8]:
# Ensure stratification worked
print("Full dataset:")
print(X['type'].value_counts(normalize=True).round(3))

print("\nTraining set:")
print(X_train['type'].value_counts(normalize=True).round(3))

print("\nTest set:")
print(X_test['type'].value_counts(normalize=True).round(3))

Full dataset:
type
1.0    0.754
0.0    0.246
Name: proportion, dtype: float64

Training set:
type
1.0    0.754
0.0    0.246
Name: proportion, dtype: float64

Test set:
type
1.0    0.754
0.0    0.246
Name: proportion, dtype: float64


We have about 75% white and 25% red wines in the full, training, and test dataset.

### Standardize Predictors

Before we start training, let's standardize our predictor values to have mean 0 and sd 1

In [9]:
means = X_train.apply(np.mean, axis = 0)
means

,0
fixed acidity,7.223167
volatile acidity,0.339449
citric acid,0.318451
residual sugar,5.430633
chlorides,0.055925
free sulfur dioxide,30.575428
total sulfur dioxide,115.957283
density,0.994701
pH,3.217400
sulphates,0.530681


In [10]:
stds = X_train.apply(np.std, axis = 0)
stds

,0
fixed acidity,1.309336
volatile acidity,0.165296
citric acid,0.144919
residual sugar,4.756470
chlorides,0.035210
free sulfur dioxide,17.558662
total sulfur dioxide,56.806273
density,0.003017
pH,0.159680
sulphates,0.147131


In [11]:
X_train = X_train.apply(lambda x: (x-np.mean(x))/np.std(x), axis = 0)
X_train

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,quality,type
4161,-0.246818,-0.117660,-0.403337,-0.658184,-0.736292,-1.114859,-0.228096,-1.206856,-0.985721,-0.752261,0.198690,0.571351
5274,-1.239687,-0.541142,-0.886366,-0.637160,-0.196679,0.650652,0.880936,-0.895248,0.204155,-0.684294,0.198690,0.571351
4708,-0.246818,-0.238655,0.286705,-0.195656,-0.054676,0.024180,1.532273,0.142339,1.143531,0.267236,-2.093415,0.571351
5008,-0.781440,-0.662137,0.217701,-0.994568,-0.139878,-0.374483,-0.087266,-1.027847,0.955656,-0.684294,-3.239468,0.571351
3840,-0.094068,-0.359649,-0.886366,0.498136,-0.452285,0.992363,1.197803,0.361127,-0.359470,-0.208529,-0.947362,0.571351
...,...,...,...,...,...,...,...,...,...,...,...,...
2968,0.135055,1.818258,1.045752,1.833159,3.183003,0.650652,1.215407,1.226335,-1.987722,-0.276495,-0.947362,0.571351
3303,-0.399567,-0.178157,0.700731,0.203800,0.030527,-0.317531,0.564070,0.264993,-0.359470,-0.548361,-0.947362,0.571351
2604,-0.475941,-0.964624,-0.265328,-0.447944,-0.338682,-0.716195,-0.439340,-0.563751,1.268781,0.743002,0.198690,0.571351
862,0.211430,0.487314,0.010689,-0.574088,0.314533,-1.342667,-1.601184,0.523561,0.141530,-0.616328,-0.947362,-1.750237


In [12]:
X_train.apply(np.mean, axis = 0)
X_train.apply(np.std, axis = 0)

,0
fixed acidity,1.0
volatile acidity,1.0
citric acid,1.0
residual sugar,1.0
chlorides,1.0
free sulfur dioxide,1.0
total sulfur dioxide,1.0
density,1.0
pH,1.0
sulphates,1.0


## Model Training

### Multiple Linear Regression (MLR) Models

We will now fit four different multiple linear regression models.
* At least one should include interaction terms
* At least one should include some polynomial terms
* Use CV to select your best MLR model

In [13]:
from sklearn.preprocessing import PolynomialFeatures

# Model 1: Basic MLR (all features)
cv_mlr1 = cross_validate(
    LinearRegression(),
    X_train,
    y_train,
    cv=5,
    scoring="neg_mean_squared_error")

# Model 2: MLR with interaction terms
X_train_interact = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False).fit_transform(X_train)
cv_mlr2 = cross_validate(
    LinearRegression(),
    X_train_interact,
    y_train,
    cv=5,
    scoring="neg_mean_squared_error")

# Model 3: MLR with polynomial terms
X_train_poly = PolynomialFeatures(degree=2, interaction_only=False, include_bias=False).fit_transform(X_train)
cv_mlr3 = cross_validate(
    LinearRegression(),
    X_train_poly,
    y_train,
    cv=5,
    scoring="neg_mean_squared_error")

# Model 4: MLR with subset of features + manual interaction and polynomial terms
X_train_custom = X_train[["residual sugar", "density", "type"]].copy()
X_train_custom["sugar_x_type"] = X_train["residual sugar"] * X_train["type"]
X_train_custom["density_sq"] = X_train["density"] ** 2
cv_mlr4 = cross_validate(
    LinearRegression(),
    X_train_custom,
    y_train,
    cv=5,
    scoring="neg_mean_squared_error")



In [14]:
# Compare RMSE across all four models
print(np.sqrt(-sum(cv_mlr1['test_score'])),
      np.sqrt(-sum(cv_mlr2['test_score'])),
      np.sqrt(-sum(cv_mlr3['test_score'])),
      np.sqrt(-sum(cv_mlr4['test_score'])))

1.163360017081903 0.9916288584882726 1.005821024745858 1.6914606656747224


In [15]:
mlr_best = LinearRegression().fit(X_train_interact, y_train)
print(mlr_best.intercept_)
print(np.array(list(zip(X_train.columns, mlr_best.coef_))))

10.342414045805445
[['fixed acidity' '0.6728307232998706']
 ['volatile acidity' '0.09448792937169068']
 ['citric acid' '0.0726211612170754']
 ['residual sugar' '1.1384644220769264']
 ['chlorides' '-0.009436266348793021']
 ['free sulfur dioxide' '-0.07740237955735757']
 ['total sulfur dioxide' '0.06205711267245362']
 ['density' '-2.0465117229471277']
 ['pH' '0.42522716948572814']
 ['sulphates' '0.15183575115244896']
 ['quality' '0.056698286690855415']
 ['type' '-0.5269500022796357']]


### LASSO model

In [19]:
# Predictor selection
features = ["fixed acidity", "volatile acidity", "residual sugar", "density", "type", "pH"]

X_train_lasso = X_train[features]
X_test_lasso = X_test[features]

lasso_mod = LassoCV(
    alphas = np.logspace(-4, 1, 100), # Tests 100 alpha values between 0.0001 and 10
    cv = 5,
    random_state = 7
)
lasso_mod.fit(X_train_lasso, y_train)

LassoCV(alphas=array([1.00000000e-04, 1.12332403e-04, 1.26185688e-04, 1.41747416e-04,
       1.59228279e-04, 1.78864953e-04, 2.00923300e-04, 2.25701972e-04,
       2.53536449e-04, 2.84803587e-04, 3.19926714e-04, 3.59381366e-04,
       4.03701726e-04, 4.53487851e-04, 5.09413801e-04, 5.72236766e-04,
       6.42807312e-04, 7.22080902e-04, 8.11130831e-04, 9.11162756e-04,
       1.02353102e-03, 1.14975700e-0...
       6.89261210e-01, 7.74263683e-01, 8.69749003e-01, 9.77009957e-01,
       1.09749877e+00, 1.23284674e+00, 1.38488637e+00, 1.55567614e+00,
       1.74752840e+00, 1.96304065e+00, 2.20513074e+00, 2.47707636e+00,
       2.78255940e+00, 3.12571585e+00, 3.51119173e+00, 3.94420606e+00,
       4.43062146e+00, 4.97702356e+00, 5.59081018e+00, 6.28029144e+00,
       7.05480231e+00, 7.92482898e+00, 8.90215085e+00, 1.00000000e+01]),
        cv=5, random_state=7)

In [20]:
# Best alpha
print(lasso_mod.alpha_)

# Intercept
print(lasso_mod.intercept_)

# Coefficients
print(np.array(list(zip(X_train_lasso.columns, lasso_mod.coef_))))

0.0006428073117284319
10.491021743313523
[['fixed acidity' '0.7653770048755917']
 ['volatile acidity' '0.05788450878891965']
 ['residual sugar' '1.1072941746157823']
 ['density' '-2.02314384117133']
 ['type' '-0.5747068517424108']
 ['pH' '0.44821896604727596']]


In [25]:
# CV RMSE at best alpha
cv_lasso_rmse = np.sqrt(np.min(np.mean(lasso_mod.mse_path_, axis=1)))
cv_lasso_rmse

np.float64(0.5482423088455753)

In [28]:
lasso_pred = lasso_mod.predict(X_test_lasso)
lasso_pred

array([33.49599927, 16.12558947, 30.75465768, ..., 19.43666751,
       17.04060018, 15.29178054])

### Ridge Regression Model


In [35]:
from sklearn.linear_model import Ridge, RidgeCV

# Predictor selection - Here we'll do slightly different than with LASSO
features2 = ["citric acid", "residual sugar", "chlorides", "quality", "type", "density"]

X_train_ridge = X_train[features2]
X_test_ridge = X_test[features2]

ridge_mod = RidgeCV(
    alphas=np.logspace(-4, 1, 100),  # range of alphas to try
    cv=5
)

ridge_mod.fit(X_train_ridge, y_train)

RidgeCV(alphas=array([1.00000000e-04, 1.12332403e-04, 1.26185688e-04, 1.41747416e-04,
       1.59228279e-04, 1.78864953e-04, 2.00923300e-04, 2.25701972e-04,
       2.53536449e-04, 2.84803587e-04, 3.19926714e-04, 3.59381366e-04,
       4.03701726e-04, 4.53487851e-04, 5.09413801e-04, 5.72236766e-04,
       6.42807312e-04, 7.22080902e-04, 8.11130831e-04, 9.11162756e-04,
       1.02353102e-03, 1.14975700e-0...
       6.89261210e-01, 7.74263683e-01, 8.69749003e-01, 9.77009957e-01,
       1.09749877e+00, 1.23284674e+00, 1.38488637e+00, 1.55567614e+00,
       1.74752840e+00, 1.96304065e+00, 2.20513074e+00, 2.47707636e+00,
       2.78255940e+00, 3.12571585e+00, 3.51119173e+00, 3.94420606e+00,
       4.43062146e+00, 4.97702356e+00, 5.59081018e+00, 6.28029144e+00,
       7.05480231e+00, 7.92482898e+00, 8.90215085e+00, 1.00000000e+01]),
        cv=5)

In [38]:
# Best alpha (tuning parameter)
print(ridge_mod.alpha_)

# Intercept
print(ridge_mod.intercept_)

# Coefficients
# Coefficients
print(np.array(list(zip(X_train_ridge.columns, ridge_mod.coef_))))

2.477076355991709
10.4910217433135
[['citric acid' '0.18932019573265182']
 ['residual sugar' '0.5734819408781667']
 ['chlorides' '-0.12370400802230236']
 ['quality' '0.18595886435949321']
 ['type' '-0.8117827994159849']
 ['density' '-1.3634554060593704']]


### Elastic Net

In [ ]:
from sklearn.linear_model import ElasticNet, ElasticNetCV

# Predictor selection
features3 = ["fixed acidity", "citric acid", "residual sugar", "density", "type", "pH", "sulphates"]

